# Batterie-Arbitrage mit linearer Optimierung in Pyomo

Dieses Notebook zeigt ein sehr einfaches Batterie-Arbitrage-Modell.

Die Idee ist:

- Die Batterie **lädt**, wenn Strom günstig ist.
- Die Batterie **entlädt**, wenn Strom teuer ist.
- Das Modell entscheidet selbst, wann Laden und Entladen wirtschaftlich sinnvoll ist.

Wir bauen das Modell in drei kurzen Kapiteln auf:

1. **Ganz simples LP:** Arbitrage ohne Zyklenkosten und ohne Selbstentladung  
2. **Zusatz 1:** Opportunitätskosten pro äquivalentem Vollzyklus  
3. **Zusatz 2:** Selbstentladung der Batterie

Als Solver verwenden wir **HiGHS** über das Python-Paket `highspy`.


## 0. Installation

Falls die Pakete noch nicht installiert sind, kann die nächste Zelle einmal ausgeführt werden.

Wichtig:  
`highspy` installiert den HiGHS-Solver direkt als Python-Paket.  
Damit ist keine separate GLPK-Installation nötig.


In [153]:
# Diese Zelle nur bei Bedarf ausführen.
# Wenn alles schon installiert ist, kann sie übersprungen werden.

%pip install -q pyomo highspy

Note: you may need to restart the kernel to use updated packages.


## 1. Pakete importieren und Preisdaten einlesen

Das Notebook liest die Preisdaten **nur** aus der Datei

`arbitrage_price_profile.csv`

Die Datei muss im gleichen Ordner liegen wie dieses Notebook.


In [154]:
import pandas as pd
import plotly.graph_objects as go
import pyomo.environ as pyo

In [1]:
# Preisdaten einlesen
price_data = pd.read_csv("../data/arbitrage_price_profile.csv")

# Wir erwarten eine Spalte mit Preisen in €/kWh
price_data.head()


NameError: name 'pd' is not defined

In [156]:
# Zeitschritt: Die Datei enthält Viertelstundenwerte.
dt_h = 0.25  # Stunden

# Für die Optimierung verwenden wir die Preise als Python-Liste.
prices = price_data["price_eur_per_kWh"].tolist()

# Anzahl der Zeitschritte
n_steps = len(prices)

# Eine Zeitachse in Stunden ist praktisch für die Darstellung.
price_data["time_h"] = range(n_steps)
price_data["time_h"] = price_data["time_h"] * dt_h

print(f"Anzahl Zeitschritte: {n_steps}")
print(f"Zeitraum: {n_steps * dt_h:.1f} Stunden")


Anzahl Zeitschritte: 672
Zeitraum: 168.0 Stunden


In [157]:
# Kurzer Blick auf das Preisprofil
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=price_data["time_h"],
        y=price_data["price_eur_per_kWh"],
        mode="lines",
        name="Preis"
    )
)

fig.update_layout(
    title="Strompreisprofil",
    xaxis_title="Zeit in h",
    yaxis_title="Preis in €/kWh",
    template="plotly_white"
)

fig.show()

## 2. Gemeinsame Batterieparameter

Diese Parameter gelten für alle drei Kapitel.

Die Werte sind bewusst einfach gewählt und können von den Studierenden verändert werden.


In [158]:
# Batteriekapazität
E_nominal_kWh = 1000.0

# Maximale Lade- und Entladeleistung
P_max_kW = 1000.0

# SOC-Grenzen
soc_min = 0.00
soc_max = 1.00
soc_start = 0.50

# Wirkungsgrad
# Zur Vereinfachung verwenden wir denselben Wirkungsgrad für Laden und Entladen.
eta = 0.95

# Abgeleitete Energiegrenzen
E_min_kWh = soc_min * E_nominal_kWh
E_max_kWh = soc_max * E_nominal_kWh
E_start_kWh = soc_start * E_nominal_kWh

print(f"E_min = {E_min_kWh:.1f} kWh")
print(f"E_max = {E_max_kWh:.1f} kWh")
print(f"E_start = {E_start_kWh:.1f} kWh")


E_min = 0.0 kWh
E_max = 1000.0 kWh
E_start = 500.0 kWh


# Kapitel 1 — Reine Arbitrage als lineares Optimierungsproblem

In diesem ersten Modell gibt es nur:

- Ladeleistung
- Entladeleistung
- Energieinhalt der Batterie
- Strompreise

Es gibt **keine Zyklenkosten** und **keine Selbstentladung**.

## Mathematische Formulierung

### Index

$t = 1,\dots,N$ ist der Zeitschritt.

### Parameter

| Symbol | Bedeutung |
|---|---|
| $\pi_t$ | Strompreis in Zeitschritt $t$ in €/kWh |
| $\Delta t$ | Länge eines Zeitschritts in h |
| $E^{\max}$ | maximale Energie der Batterie in kWh |
| $E^{\min}$ | minimale Energie der Batterie in kWh |
| $P^{\max}$ | maximale Lade- und Entladeleistung in kW |
| $\eta$ | Wirkungsgrad |
| $E_0$ | Anfangsenergie der Batterie in kWh |

### Entscheidungsvariablen

| Variable | Bedeutung |
|---|---|
| $P^{\mathrm{ch}}_t$ | Ladeleistung in kW |
| $P^{\mathrm{dis}}_t$ | Entladeleistung in kW |
| $E_t$ | Energieinhalt der Batterie nach Zeitschritt $t$ in kWh |

### Zielfunktion

Wir maximieren den Arbitragegewinn:

$$
\max
\sum_{t=1}^{N}
\pi_t
\left(
P^{\mathrm{dis}}_t
-
P^{\mathrm{ch}}_t
\right)
\Delta t
$$

Der erste Teil ist der Erlös durch Entladen.  
Der zweite Teil sind die Kosten durch Laden.

### Nebenbedingungen

Leistungsgrenzen:

$$
0 \leq P^{\mathrm{ch}}_t \leq P^{\max}
\qquad \forall t
$$

$$
0 \leq P^{\mathrm{dis}}_t \leq P^{\max}
\qquad \forall t
$$

Gemeinsame Wechselrichter- bzw. Netzanschluss-Leistungsgrenze:

$$
P^{\mathrm{ch}}_t + P^{\mathrm{dis}}_t \leq P^{\max}
\qquad \forall t
$$

Energiegrenzen:

$$
E^{\min} \leq E_t \leq E^{\max}
\qquad \forall t
$$

Energiebilanz der Batterie:

$$
E_t
=
E_{t-1}
+
\eta P^{\mathrm{ch}}_t \Delta t
-
\frac{1}{\eta} P^{\mathrm{dis}}_t \Delta t
\qquad \forall t
$$

Gleicher Anfangs- und Endzustand:

$$
E_N = E_0
$$

Diese Bedingung verhindert, dass die Batterie am Ende einfach leergefahren wird.


In [159]:
# ------------------------------------------------------------
# Kapitel 1: Modell erstellen
# ------------------------------------------------------------

model1 = pyo.ConcreteModel()

# Entscheidungsraum: Indizes für Zeitschritte
model1.P = pyo.RangeSet(0, n_steps - 1)  # Zeitschritte für Leistung
model1.E = pyo.RangeSet(0, n_steps)      # Zeitschritte für Energie

# Entscheidungsvariablen
model1.p_charge = pyo.Var(model1.P, bounds=(0, P_max_kW))
model1.p_discharge = pyo.Var(model1.P, bounds=(0, P_max_kW))
model1.energy = pyo.Var(model1.E, bounds=(E_min_kWh, E_max_kWh))

# Anfangsenergie
model1.start_energy = pyo.Constraint(
    expr=model1.energy[0] == E_start_kWh
)

# Energiebilanz
def energy_balance_rule_1(model, t):
    return model.energy[t + 1] == (model.energy[t] 
        + eta * model.p_charge[t] * dt_h - (1 / eta) * model.p_discharge[t] * dt_h)

model1.energy_balance = pyo.Constraint(model1.P, rule=energy_balance_rule_1)

# Gemeinsame Leistungsgrenze
def total_power_rule_1(model, t):
    return model.p_charge[t] + model.p_discharge[t] <= P_max_kW

model1.total_power_limit = pyo.Constraint(model1.P, rule=total_power_rule_1)

# Endenergie = Anfangsenergie
model1.end_energy = pyo.Constraint(
    expr=model1.energy[n_steps] == E_start_kWh
)

# Zielfunktion: Arbitragegewinn
model1.objective = pyo.Objective(
    expr=sum(prices[t] * (model1.p_discharge[t] - model1.p_charge[t]) * dt_h for t in model1.P),
    sense=pyo.maximize
)


In [160]:
# ------------------------------------------------------------
# Kapitel 1: Modell lösen
# ------------------------------------------------------------

solver = pyo.SolverFactory("appsi_highs")

if not solver.available():
    raise RuntimeError(
        "HiGHS ist nicht verfügbar. "
        "Bitte installiere den Solver mit: pip install highspy"
    )


def print_solver_status(result):
    """
    Pyomo kann je nach Solver-Schnittstelle unterschiedliche Ergebnisobjekte liefern.
    Bei SolverFactory("appsi_highs") ist die termination condition in vielen
    Installationen unter result.solver.termination_condition gespeichert.
    """
    if hasattr(result, "termination_condition"):
        termination = result.termination_condition
    else:
        termination = result.solver.termination_condition

    print("Solver-Status:", termination)


result1 = solver.solve(model1)

print_solver_status(result1)
print(f"Zielfunktionswert: {pyo.value(model1.objective):.2f} €")


Solver-Status: optimal
Zielfunktionswert: 2523.22 €


In [161]:
# ------------------------------------------------------------
# Kapitel 1: Ergebnisse aus dem Modell holen
# ------------------------------------------------------------

result_simple = pd.DataFrame({
    "time_h": price_data["time_h"],
    "price_eur_per_kWh": price_data["price_eur_per_kWh"],
    "p_charge_kW": [pyo.value(model1.p_charge[t]) for t in range(n_steps)],
    "p_discharge_kW": [pyo.value(model1.p_discharge[t]) for t in range(n_steps)],
    "energy_kWh": [pyo.value(model1.energy[t]) for t in range(n_steps)]
})

result_simple["charged_energy_kWh"] = result_simple["p_charge_kW"] * dt_h
result_simple["discharged_energy_kWh"] = result_simple["p_discharge_kW"] * dt_h

result_simple["market_profit_eur"] = (
    result_simple["price_eur_per_kWh"]
    * (
        result_simple["discharged_energy_kWh"]
        - result_simple["charged_energy_kWh"]
    )
)

result_simple.head()


,time_h,price_eur_per_kWh,p_charge_kW,p_discharge_kW,energy_kWh,charged_energy_kWh,discharged_energy_kWh,market_profit_eur
0,0.00,0.042877,0.000000,0.0,500.000000,0.000000,0.0,0.000000
1,0.25,0.041197,110.803324,0.0,500.000000,27.700831,0.0,-1.141191
2,0.50,0.057489,0.000000,1000.0,526.315789,0.000000,250.0,14.372250
3,0.75,0.061953,0.000000,1000.0,263.157895,0.000000,250.0,15.488250
4,1.00,0.031675,1000.000000,0.0,0.000000,250.000000,0.0,-7.918750


In [162]:
# Kennzahlen für Kapitel 1
profit_simple = result_simple["market_profit_eur"].sum()
charged_simple = result_simple["charged_energy_kWh"].sum()
discharged_simple = result_simple["discharged_energy_kWh"].sum()
cycles_simple = discharged_simple / E_nominal_kWh

print(f"Gewinn:                  {profit_simple:10.2f} €")
print(f"Geladene Energie:        {charged_simple:10.2f} kWh")
print(f"Entladene Energie:       {discharged_simple:10.2f} kWh")
print(f"Äquivalente Vollzyklen:  {cycles_simple:10.2f}")


Gewinn:                     2523.22 €
Geladene Energie:          61796.03 kWh
Entladene Energie:         55770.92 kWh
Äquivalente Vollzyklen:       55.77


In [163]:
# Plot für Kapitel 1
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=result_simple["time_h"],
        y=result_simple["price_eur_per_kWh"],
        mode="lines",
        name="Preis",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=result_simple["time_h"],
        y=result_simple["energy_kWh"],
        mode="lines",
        name="Energieinhalt",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Kapitel 1: Reine Arbitrage",
    xaxis=dict(title="Zeit in h"),
    yaxis=dict(title="Preis in €/kWh"),
    yaxis2=dict(
        title="Energieinhalt in kWh",
        overlaying="y",
        side="right"
    ),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# Kapitel 2 — Zusatz 1: Opportunitätskosten pro Vollzyklus

Nun ergänzen wir Kosten für die Nutzung der Batterie.

Die Idee: Jeder äquivalente Vollzyklus verursacht Opportunitätskosten, zum Beispiel durch Alterung oder entgangene andere Nutzungen.

Wir rechnen hier sehr einfach:

$$
n_{cycles}
=
\frac{
\sum_{t=1}^{N}
P^{\mathrm{dis}}_t \Delta t
}{
E^{\mathrm{nom}}
}
$$

Das bedeutet:  
Wenn eine Batterie mit $1000\,\mathrm{kWh}$ Kapazität insgesamt $1000\,\mathrm{kWh}$ entlädt, entspricht das einem äquivalenten Vollzyklus.

## Neue Zielfunktion

Die Nebenbedingungen bleiben gleich wie in Kapitel 1.

Nur die Zielfunktion wird erweitert:

$$
\max
\sum_{t=1}^{N}
\pi_t
\left(
P^{\mathrm{dis}}_t
-
P^{\mathrm{ch}}_t
\right)
\Delta t
-
c^{\mathrm{cycle}}
\cdot
n_{cycles}
$$

Dabei ist $c^{\mathrm{cycle}}$ der Kostenwert in € pro äquivalentem Vollzyklus.

Weil $c^{\mathrm{cycle}}$ und $E^{\mathrm{nom}}$ Konstanten sind, bleibt das Modell linear.


In [164]:
# Opportunitätskosten pro äquivalentem Vollzyklus
cycle_cost_eur_per_full_cycle = 5.0


In [165]:
# ------------------------------------------------------------
# Kapitel 2: Modell mit Zyklenkosten erstellen
# ------------------------------------------------------------

model2 = pyo.ConcreteModel()

model2.T = pyo.RangeSet(0, n_steps - 1)
model2.S = pyo.RangeSet(0, n_steps)

model2.p_charge = pyo.Var(model2.T, bounds=(0, P_max_kW))
model2.p_discharge = pyo.Var(model2.T, bounds=(0, P_max_kW))
model2.energy = pyo.Var(model2.S, bounds=(E_min_kWh, E_max_kWh))

model2.start_energy = pyo.Constraint(
    expr=model2.energy[0] == E_start_kWh
)

def energy_balance_rule_2(model, t):
    return model.energy[t + 1] == (
        model.energy[t]
        + eta * model.p_charge[t] * dt_h
        - (1 / eta) * model.p_discharge[t] * dt_h
    )

model2.energy_balance = pyo.Constraint(model2.T, rule=energy_balance_rule_2)

def total_power_rule_2(model, t):
    return model.p_charge[t] + model.p_discharge[t] <= P_max_kW

model2.total_power_limit = pyo.Constraint(model2.T, rule=total_power_rule_2)

model2.end_energy = pyo.Constraint(
    expr=model2.energy[n_steps] == E_start_kWh
)

# Markterlös
market_profit_2 = sum(
    prices[t] * (model2.p_discharge[t] - model2.p_charge[t]) * dt_h
    for t in model2.T
)

# Äquivalente Vollzyklen
equivalent_full_cycles_2 = sum(
    model2.p_discharge[t] * dt_h
    for t in model2.T
) / E_nominal_kWh

# Zielfunktion mit Zyklenkosten
model2.objective = pyo.Objective(
    expr=market_profit_2
    - cycle_cost_eur_per_full_cycle * equivalent_full_cycles_2,
    sense=pyo.maximize
)


In [166]:
# Kapitel 2: Modell lösen
result2 = solver.solve(model2)

print_solver_status(result2)
print(f"Zielfunktionswert: {pyo.value(model2.objective):.2f} €")


Solver-Status: optimal
Zielfunktionswert: 2273.10 €


In [167]:
# Kapitel 2: Ergebnisse
result_cycle_costs = pd.DataFrame({
    "time_h": price_data["time_h"],
    "price_eur_per_kWh": price_data["price_eur_per_kWh"],
    "p_charge_kW": [pyo.value(model2.p_charge[t]) for t in range(n_steps)],
    "p_discharge_kW": [pyo.value(model2.p_discharge[t]) for t in range(n_steps)],
    "energy_kWh": [pyo.value(model2.energy[t]) for t in range(n_steps)]
})

result_cycle_costs["charged_energy_kWh"] = result_cycle_costs["p_charge_kW"] * dt_h
result_cycle_costs["discharged_energy_kWh"] = result_cycle_costs["p_discharge_kW"] * dt_h

result_cycle_costs["market_profit_eur"] = (
    result_cycle_costs["price_eur_per_kWh"]
    * (
        result_cycle_costs["discharged_energy_kWh"]
        - result_cycle_costs["charged_energy_kWh"]
    )
)

result_cycle_costs["cycle_cost_eur"] = (
    cycle_cost_eur_per_full_cycle
    * result_cycle_costs["discharged_energy_kWh"]
    / E_nominal_kWh
)

result_cycle_costs["profit_after_cycle_costs_eur"] = (
    result_cycle_costs["market_profit_eur"]
    - result_cycle_costs["cycle_cost_eur"]
)

result_cycle_costs.head()


,time_h,price_eur_per_kWh,p_charge_kW,p_discharge_kW,energy_kWh,charged_energy_kWh,discharged_energy_kWh,market_profit_eur,cycle_cost_eur,profit_after_cycle_costs_eur
0,0.00,0.042877,0.000000,0.0,500.000000,0.000000,0.0,0.000000,0.00,0.000000
1,0.25,0.041197,110.803324,0.0,500.000000,27.700831,0.0,-1.141191,0.00,-1.141191
2,0.50,0.057489,0.000000,1000.0,526.315789,0.000000,250.0,14.372250,1.25,13.122250
3,0.75,0.061953,0.000000,1000.0,263.157895,0.000000,250.0,15.488250,1.25,14.238250
4,1.00,0.031675,1000.000000,0.0,0.000000,250.000000,0.0,-7.918750,0.00,-7.918750


In [168]:
# Kennzahlen für Kapitel 2
market_profit_cycle = result_cycle_costs["market_profit_eur"].sum()
cycle_costs_total = result_cycle_costs["cycle_cost_eur"].sum()
profit_cycle = result_cycle_costs["profit_after_cycle_costs_eur"].sum()
cycles_cycle = result_cycle_costs["discharged_energy_kWh"].sum() / E_nominal_kWh

print(f"Marktgewinn vor Zyklenkosten: {market_profit_cycle:10.2f} €")
print(f"Äquivalente Vollzyklen:       {cycles_cycle:10.2f}")
print(f"Zyklenkosten:                 {cycle_costs_total:10.2f} €")
print(f"Gewinn nach Zyklenkosten:     {profit_cycle:10.2f} €")


Marktgewinn vor Zyklenkosten:    2490.20 €
Äquivalente Vollzyklen:            43.42
Zyklenkosten:                     217.11 €
Gewinn nach Zyklenkosten:        2273.10 €


In [169]:
# Plot für Kapitel 2
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=result_cycle_costs["time_h"],
        y=result_cycle_costs["price_eur_per_kWh"],
        mode="lines",
        name="Preis",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=result_cycle_costs["time_h"],
        y=result_cycle_costs["energy_kWh"],
        mode="lines",
        name="Energieinhalt",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Kapitel 2: Arbitrage mit Zyklenkosten",
    xaxis=dict(title="Zeit in h"),
    yaxis=dict(title="Preis in €/kWh"),
    yaxis2=dict(
        title="Energieinhalt in kWh",
        overlaying="y",
        side="right"
    ),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# Kapitel 3 — Zusatz 2: Selbstentladung

Nun ergänzen wir zusätzlich eine einfache Selbstentladung.

Die Batterie verliert pro Tag einen kleinen Anteil ihres Energieinhalts.

Wir definieren:

$$
\alpha
=
(1-\lambda_{\mathrm{day}})^{\Delta t / 24}
$$

Dabei ist:

- $\lambda_{\mathrm{day}}$ die Selbstentladung pro Tag
- $\alpha$ der verbleibende Energieanteil pro Zeitschritt

## Neue Energiebilanz

Die Zielfunktion bleibt wie in Kapitel 2 mit Zyklenkosten.

Die Energiebilanz wird zu:

$$
E_t
=
\alpha E_{t-1}
+
\eta P^{\mathrm{ch}}_t \Delta t
-
\frac{1}{\eta} P^{\mathrm{dis}}_t \Delta t
\qquad \forall t
$$

Alle Terme sind weiterhin linear, weil $\alpha$ ein konstanter Parameter ist.


In [170]:
# Selbstentladung pro Tag
# Beispiel: 0.002 bedeutet 0,2 % pro Tag.
# Setze den Wert auf 0.0, wenn keine Selbstentladung berücksichtigt werden soll.
self_discharge_per_day = 0.002

# Umrechnung auf einen Zeitschritt
alpha = (1 - self_discharge_per_day) ** (dt_h / 24)

print(f"Verbleibender Energieanteil pro Zeitschritt alpha = {alpha:.8f}")


Verbleibender Energieanteil pro Zeitschritt alpha = 0.99997915


In [171]:
# ------------------------------------------------------------
# Kapitel 3: Modell mit Zyklenkosten und Selbstentladung
# ------------------------------------------------------------

model3 = pyo.ConcreteModel()

model3.T = pyo.RangeSet(0, n_steps - 1)
model3.S = pyo.RangeSet(0, n_steps)

model3.p_charge = pyo.Var(model3.T, bounds=(0, P_max_kW))
model3.p_discharge = pyo.Var(model3.T, bounds=(0, P_max_kW))
model3.energy = pyo.Var(model3.S, bounds=(E_min_kWh, E_max_kWh))

model3.start_energy = pyo.Constraint(
    expr=model3.energy[0] == E_start_kWh
)

def energy_balance_rule_3(model, t):
    return model.energy[t + 1] == (
        alpha * model.energy[t]
        + eta * model.p_charge[t] * dt_h
        - (1 / eta) * model.p_discharge[t] * dt_h
    )

model3.energy_balance = pyo.Constraint(model3.T, rule=energy_balance_rule_3)

def total_power_rule_3(model, t):
    return model.p_charge[t] + model.p_discharge[t] <= P_max_kW

model3.total_power_limit = pyo.Constraint(model3.T, rule=total_power_rule_3)

model3.end_energy = pyo.Constraint(
    expr=model3.energy[n_steps] == E_start_kWh
)

market_profit_3 = sum(
    prices[t] * (model3.p_discharge[t] - model3.p_charge[t]) * dt_h
    for t in model3.T
)

equivalent_full_cycles_3 = sum(
    model3.p_discharge[t] * dt_h
    for t in model3.T
) / E_nominal_kWh

model3.objective = pyo.Objective(
    expr=market_profit_3
    - cycle_cost_eur_per_full_cycle * equivalent_full_cycles_3,
    sense=pyo.maximize
)


In [172]:
# Kapitel 3: Modell lösen
result3 = solver.solve(model3)

print_solver_status(result3)
print(f"Zielfunktionswert: {pyo.value(model3.objective):.2f} €")


Solver-Status: optimal
Zielfunktionswert: 2272.80 €


In [173]:
# Kapitel 3: Ergebnisse
result_cycle_costs_self_discharge = pd.DataFrame({
    "time_h": price_data["time_h"],
    "price_eur_per_kWh": price_data["price_eur_per_kWh"],
    "p_charge_kW": [pyo.value(model3.p_charge[t]) for t in range(n_steps)],
    "p_discharge_kW": [pyo.value(model3.p_discharge[t]) for t in range(n_steps)],
    "energy_kWh": [pyo.value(model3.energy[t]) for t in range(n_steps)]
})

result_cycle_costs_self_discharge["charged_energy_kWh"] = (
    result_cycle_costs_self_discharge["p_charge_kW"] * dt_h
)

result_cycle_costs_self_discharge["discharged_energy_kWh"] = (
    result_cycle_costs_self_discharge["p_discharge_kW"] * dt_h
)

result_cycle_costs_self_discharge["market_profit_eur"] = (
    result_cycle_costs_self_discharge["price_eur_per_kWh"]
    * (
        result_cycle_costs_self_discharge["discharged_energy_kWh"]
        - result_cycle_costs_self_discharge["charged_energy_kWh"]
    )
)

result_cycle_costs_self_discharge["cycle_cost_eur"] = (
    cycle_cost_eur_per_full_cycle
    * result_cycle_costs_self_discharge["discharged_energy_kWh"]
    / E_nominal_kWh
)

result_cycle_costs_self_discharge["profit_after_cycle_costs_eur"] = (
    result_cycle_costs_self_discharge["market_profit_eur"]
    - result_cycle_costs_self_discharge["cycle_cost_eur"]
)

result_cycle_costs_self_discharge.head()


,time_h,price_eur_per_kWh,p_charge_kW,p_discharge_kW,energy_kWh,charged_energy_kWh,discharged_energy_kWh,market_profit_eur,cycle_cost_eur,profit_after_cycle_costs_eur
0,0.00,0.042877,0.000000,0.0,500.000000,0.000000,0.0,0.000000,0.00,0.000000
1,0.25,0.041197,110.960452,0.0,499.989573,27.740113,0.0,-1.142809,0.00,-1.142809
2,0.50,0.057489,0.000000,1000.0,526.332254,0.000000,250.0,14.372250,1.25,13.122250
3,0.75,0.061953,0.000000,1000.0,263.163383,0.000000,250.0,15.488250,1.25,14.238250
4,1.00,0.031675,1000.000000,0.0,0.000000,250.000000,0.0,-7.918750,0.00,-7.918750


In [174]:
# Kennzahlen für Kapitel 3
market_profit_self = result_cycle_costs_self_discharge["market_profit_eur"].sum()
cycle_costs_self = result_cycle_costs_self_discharge["cycle_cost_eur"].sum()
profit_self = result_cycle_costs_self_discharge["profit_after_cycle_costs_eur"].sum()
cycles_self = (
    result_cycle_costs_self_discharge["discharged_energy_kWh"].sum()
    / E_nominal_kWh
)

print(f"Marktgewinn vor Zyklenkosten: {market_profit_self:10.2f} €")
print(f"Äquivalente Vollzyklen:       {cycles_self:10.2f}")
print(f"Zyklenkosten:                 {cycle_costs_self:10.2f} €")
print(f"Gewinn nach Zyklenkosten:     {profit_self:10.2f} €")


Marktgewinn vor Zyklenkosten:    2489.89 €
Äquivalente Vollzyklen:            43.42
Zyklenkosten:                     217.09 €
Gewinn nach Zyklenkosten:        2272.80 €


In [175]:
# Plot für Kapitel 3
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=result_cycle_costs_self_discharge["time_h"],
        y=result_cycle_costs_self_discharge["price_eur_per_kWh"],
        mode="lines",
        name="Preis",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=result_cycle_costs_self_discharge["time_h"],
        y=result_cycle_costs_self_discharge["energy_kWh"],
        mode="lines",
        name="Energieinhalt",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Kapitel 3: Arbitrage mit Zyklenkosten und Selbstentladung",
    xaxis=dict(title="Zeit in h"),
    yaxis=dict(title="Preis in €/kWh"),
    yaxis2=dict(
        title="Energieinhalt in kWh",
        overlaying="y",
        side="right"
    ),
    template="plotly_white",
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# 4. Vergleich der drei Modelle

Zum Schluss vergleichen wir die drei Modellvarianten.


In [176]:
summary = pd.DataFrame({
    "Modell": [
        "1: reine Arbitrage",
        "2: mit Zyklenkosten",
        "3: mit Zyklenkosten + Selbstentladung"
    ],
    "Marktgewinn_vor_Zyklenkosten_EUR": [
        profit_simple,
        market_profit_cycle,
        market_profit_self
    ],
    "Zyklenkosten_EUR": [
        0.0,
        cycle_costs_total,
        cycle_costs_self
    ],
    "Gewinn_nach_Zyklenkosten_EUR": [
        profit_simple,
        profit_cycle,
        profit_self
    ],
    "AequiVollzyklen": [
        cycles_simple,
        cycles_cycle,
        cycles_self
    ]
})

summary


,Modell,Marktgewinn_vor_Zyklenkosten_EUR,Zyklenkosten_EUR,Gewinn_nach_Zyklenkosten_EUR,AequiVollzyklen
0,1: reine Arbitrage,2523.221499,0.000000,2523.221499,55.770918
1,2: mit Zyklenkosten,2490.202254,217.105388,2273.096866,43.421078
2,3: mit Zyklenkosten + Selbstentladung,2489.893225,217.088685,2272.804540,43.417737


In [177]:
# Ergebnisse optional speichern
result_simple.to_csv("result_1_reine_arbitrage.csv", index=False)
result_cycle_costs.to_csv("result_2_mit_zyklenkosten.csv", index=False)
result_cycle_costs_self_discharge.to_csv(
    "result_3_mit_zyklenkosten_und_selbstentladung.csv",
    index=False
)
summary.to_csv("summary_arbitrage_models.csv", index=False)

print("Ergebnisdateien wurden gespeichert.")


Ergebnisdateien wurden gespeichert.


## Hinweise für Studierende

Zum Experimentieren können vor allem diese Werte geändert werden:

- `E_nominal_kWh`
- `P_max_kW`
- `eta`
- `cycle_cost_eur_per_full_cycle`
- `self_discharge_per_day`

Wichtig:  
Dieses Notebook verwendet ein **lineares Optimierungsmodell**.  
Es enthält keine Binärvariablen. Daher ist es bewusst einfacher als ein vollständiges Betriebsmodell mit striktem Entweder-Laden-oder-Entladen.
